# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OjaswiGautam/FlyrankAI/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Per the training-honest-models skill file's own method-selection table — "grouping items → K-Means (pick k with silhouette), then NAME clusters after inspecting them → unsupervised needs human naming" — K-Means is the directly recommended method for this exact question shape, and it matches Lane 3's task framing established back in w02: this is unsupervised archetype discovery, not classification or ranking, so there is no label to fit against and no precision@K to chase.

Why K-Means specifically, over the toolkit's other options:

*   Not Logistic Regression / Random Forest / Gradient Boosting — these require a supervised label. Lane 3 has none, by design (w02 established this explicitly: inventing a proxy label just to use a supervised method would mean fabricating the exact "ground truth" this lane is supposed to discover, not assume).

*   Not correlation/signal analysis alone — useful as a diagnostic (and we used it in w04's signal checks), but it answers "does X relate to Y," not "what groups exist." It doesn't produce archetypes.

*   K-Means over other clustering methods (e.g., GMM, HDBSCAN) — K-Means is the method the skill file names directly for this task shape, it produces hard, interpretable cluster assignments (a page belongs to exactly one archetype, which maps cleanly to a review workflow), and it lets us pick k transparently via silhouette rather than relying on a density parameter that's harder to justify to a non-technical reviewer. Alternative methods remain a reasonable future extension (noted as a stretch goal from the recent review), not a requirement for this baseline model.

*   Fits the feature frame directly — the 5-feature frame from w03 (gsc_impressions, gsc_avg_position, gsc_clicks, content_age_days, word_count), all numeric after preprocessing (log-transform, imputation, missingness flags, scaling), is exactly the kind of continuous feature space K-Means is built for — a distance-based method needs numeric, scaled inputs, which this frame provides after the preprocessing work already done and verified.

What this method can and cannot do, stated up front: K-Means will produce a fixed number of behavioral groups based on distance in the scaled feature space — it does not predict, does not rank by priority, and does not explain why a page behaves a certain way. It answers "what recurring patterns exist," consistent with the lane's original research question from w01, not "what should be done about it" — that interpretive step happens afterward, by inspecting real cluster profiles and examples, not by the algorithm itself.





## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Per the lane guide's Validation Rules section — "client/group holdout, when pages from the same client may share patterns the model could memorize" — this project uses a client-grouped split: GroupShuffleSplit on client_hash_id, 75/25, producing 35 train clients / 133,474 rows and 12 validation clients / 43,264 rows, with zero client overlap between the two sets (verified directly in output, not assumed).

Why grouped-by-client, over the toolkit's other split options:

*   Not a plain random row split — pages from the same client are not independent observations. A single client's content likely shares a CMS template, an editorial word-count convention, and a consistent GSC tracking setup, so pages from the same client will naturally sit close together in feature space regardless of any real archetype. A random split would let near-duplicate client patterns leak into both train and validation, inflating the silhouette and Davies-Bouldin scores by rewarding the model for memorizing a client's "house style" rather than discovering a genuine cross-client behavioral pattern. This is the exact failure mode the leakage checklist warns against: "are duplicate or related rows split across train and test in a way that makes the test too easy?"

*  Not a time-aware split — a time-aware train/test split is the right design when the question is about predicting a future outcome from a prior window (the lane guide reserves this explicitly for the Growth/Recovery/Momentum freestyle direction). This notebook's population is a single fixed window — March 2026, gsc_data_available=TRUE — aggregated to one row per (client, content) with no forward-looking target. Lane 3's question, established in w01/w02, is "what recurring behavioral archetypes exist," not "what will happen next" — there is no future window to hold out, so a time split would be solving a problem this lane doesn't have.

*  Client-grouped is what the modeling population's own structure demands — the sanity checks already run in this notebook (Cell 14's per-cluster top-client-share numbers: 14.0%–39.3%) confirm that client identity is a real, measurable source of correlation within the feature space. A validation design that ignores this would be validating against noise it already knows exists.

What this split can and cannot prove: a client-grouped holdout tells us whether the cluster structure generalizes to clients the model has never seen — which is the right bar for an archetype system meant to apply across FlyRank's whole client base, not just the training clients. It does not tell us whether the structure is stable over time for a given client (that would need a time-aware design layered on top, and is out of scope for this single-month, cross-sectional clustering question). The val-set silhouette (0.3568) and Davies-Bouldin (0.9087) reported for k=4 are therefore honestly comparable to the train-set numbers precisely because the split guarantees no client-level shortcut was available to either side.





## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The analysis below demonstrates the shift from our Week 4 rule-based baseline (a targeted operational queue) to our Week 5 descriptive model (a structural grouping).
*   **Model:** The K-Means model applies to the complete 176k GSC-available row frame. After testing multiple $K$ values with a full stack of unsupervised receipts (Silhouette, Calinski-Harabasz, Davies-Bouldin, and cluster balance), $K=4$ emerges as the most stable descriptive structure. It achieves a validation silhouette score of ~0.35 on a robust, client-grouped split.
*   **Baseline Overlay:** Rather than comparing silhouette score to operational coverage (which are fundamentally different metrics), we treat the Week 4 baseline as a *post-hoc overlay*. When joined onto the cluster profiles, we see the baseline flags are heavily concentrated in a single high-traffic cluster, while remaining blind to other archetypes.
The profile confirms that while the baseline successfully prioritizes a specific problem, the ML model uncovers the broader landscape of our content behavior.

In [1]:
import os
from google.colab import userdata
# Manually push your Colab secret into the standard environment variables
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")


In [2]:
import os
import duckdb
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, RobustScaler
from sklearn.cluster import KMeans

# Deepnote-safe secret handling: check presence only; never print secret values.
required_env_vars = ["HF_TOKEN"]
missing_env_vars = [name for name in required_env_vars if not os.environ.get(name)]
if missing_env_vars:
    raise RuntimeError(
        "Missing required environment variables: " + ", ".join(missing_env_vars)
        + ". Add them in your environment settings."
    )

# -----------------------------
# 1. Load Data & Create Base Frame
# -----------------------------
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")

TABLE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
DIM = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

# Reverted to exact query from Model.ipynb (no daily date column used)
modeling_frame = con.sql(f"""
    WITH scoped AS (
        SELECT client_hash_id, content_hash_id, gsc_impressions, gsc_clicks, gsc_avg_position
        FROM read_parquet('{TABLE}')
        WHERE gsc_data_available = TRUE
    ),
    agg AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS gsc_impressions,
               SUM(gsc_clicks) AS gsc_clicks,
               SUM(gsc_impressions * gsc_avg_position) / NULLIF(SUM(gsc_impressions), 0) AS gsc_avg_position
        FROM scoped
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT a.*, d.content_created_date, d.word_count
    FROM agg a
    LEFT JOIN read_parquet('{DIM}') d
      ON a.client_hash_id = d.client_hash_id
     AND a.content_hash_id = d.content_hash_id
""").df()

modeling_frame['content_age_days'] = (pd.Timestamp('2026-03-31') - pd.to_datetime(modeling_frame['content_created_date'])).dt.days
modeling_frame['gsc_ctr'] = modeling_frame['gsc_clicks'] / modeling_frame['gsc_impressions'].replace(0, np.nan)

df = modeling_frame.copy()

# -----------------------------
# 2. Data-contract checks (Receipts)
# -----------------------------
key_cols = ["client_hash_id", "content_hash_id"]
duplicate_keys = df.duplicated(key_cols).sum()
if duplicate_keys:
    raise ValueError(f"Expected one row per client/content pair, but found {duplicate_keys} duplicate composite keys.")

# Safe date check using the reviewer's fallback logic
date_col = "date"
if date_col in df.columns:
    min_date = pd.to_datetime(df[date_col]).min()
    max_date = pd.to_datetime(df[date_col]).max()
    distinct_days = pd.to_datetime(df[date_col]).dt.date.nunique()
    if min_date < pd.Timestamp("2026-03-01") or max_date > pd.Timestamp("2026-03-31"):
        raise ValueError(f"Observation window out of bounds: {min_date} to {max_date}")
    print(f"Observation Window: min_date={min_date.date()}, max_date={max_date.date()}, distinct_days={distinct_days}")
else:
    print("No row-level date column found in `modeling_frame`; verify date-window receipts in the SQL/build cell.")

# Banned leakage fields check
forbidden_fields = {"health_score", "priority_score", "action_type", "action_label", "baseline_score", "baseline_flag"}
numeric_features = ["gsc_clicks", "gsc_impressions", "gsc_ctr", "gsc_avg_position", "word_count", "content_age_days"]
leaked_inputs = sorted(set(numeric_features) & forbidden_fields)
if leaked_inputs:
    raise ValueError(f"Forbidden leakage fields included as model inputs: {leaked_inputs}")

# Treat zero average position as missing
df["gsc_avg_position_zero_flag"] = (df["gsc_avg_position"] == 0).astype(int)
df.loc[df["gsc_avg_position"] == 0, "gsc_avg_position"] = np.nan

contract_summary = pd.DataFrame({
    "column": numeric_features + ["gsc_avg_position_zero_flag"],
    "null_rate": [df[col].isna().mean() for col in numeric_features + ["gsc_avg_position_zero_flag"]],
    "zero_rate": [(df[col] == 0).mean() if pd.api.types.is_numeric_dtype(df[col]) else np.nan for col in numeric_features + ["gsc_avg_position_zero_flag"]],
})
print("\nData-contract summary for model inputs:")
print(contract_summary.to_string(index=False))

# -----------------------------
# 3. Split rows by client
# -----------------------------
# Client-grouped split to prevent the model from memorizing client templates
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, valid_idx = next(gss.split(df, groups=df['client_hash_id']))
train_df = df.iloc[train_idx].reset_index(drop=True)
valid_df = df.iloc[valid_idx].reset_index(drop=True)

log_features = ["gsc_clicks", "gsc_impressions", "word_count", "content_age_days"]
other_numeric_features = ["gsc_ctr", "gsc_avg_position", "gsc_avg_position_zero_flag"]
model_features = log_features + other_numeric_features

log_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scaler", RobustScaler()),
])
robust_numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)), # handles zero flag missingness elegantly
    ("scaler", RobustScaler()),
])
preprocessor = ColumnTransformer(transformers=[
    ("log", log_transformer, log_features),
    ("robust", robust_numeric_transformer, other_numeric_features),
])

X_train = preprocessor.fit_transform(train_df[model_features])
X_valid = preprocessor.transform(valid_df[model_features])

# -----------------------------
# 4. K sweep with multiple unsupervised receipts
# -----------------------------
k_rows = []
fitted_models = {}
for k in range(3, 8):
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    train_labels = km.fit_predict(X_train)
    valid_labels = km.predict(X_valid)
    fitted_models[k] = km

    valid_cluster_share = pd.Series(valid_labels).value_counts(normalize=True).sort_index()
    k_rows.append({
        "k": k,
        "train_silhouette": round(silhouette_score(X_train, train_labels, sample_size=10000, random_state=42), 4),
        "valid_silhouette": round(silhouette_score(X_valid, valid_labels, sample_size=10000, random_state=42), 4),
        "valid_calinski_harabasz": round(calinski_harabasz_score(X_valid, valid_labels), 1),
        "valid_davies_bouldin": round(davies_bouldin_score(X_valid, valid_labels), 4),
        "smallest_valid_cluster_share": round(valid_cluster_share.min(), 4),
    })

print("\nK-selection summary:")
print(pd.DataFrame(k_rows).to_string(index=False))

# Setting k=4 based on strong silhouette and davies-bouldin validation scores
final_k = 4
final_model = fitted_models[final_k]
df["cluster_id"] = final_model.predict(preprocessor.transform(df[model_features]))

# -----------------------------
# 5. Original-scale cluster profiles
# -----------------------------
profile = (
    df.groupby("cluster_id")
    .agg(
        rows=("content_hash_id", "size"),
        clients=("client_hash_id", "nunique"),
        median_clicks=("gsc_clicks", "median"),
        median_impressions=("gsc_impressions", "median"),
        median_ctr=("gsc_ctr", "median"),
        median_position=("gsc_avg_position", "median"),
        median_word_count=("word_count", "median")
    )
    .assign(row_share=lambda x: (x["rows"] / len(df)).round(4))
    .reset_index()
)
print("\nOriginal-scale cluster profile:")
print(profile.to_string(index=False))

# -----------------------------
# 6. Baseline Overlay
# -----------------------------
# Recompute week 4 baseline (TITLE_META_CTR_FIX) explicitly on the modeling frame
baseline_mask = (df['gsc_impressions'] >= 100) & (df['gsc_avg_position'] > 0)
df['position_tier'] = np.where(df['gsc_avg_position'] <= 3, 'pos_1_3',
                      np.where(df['gsc_avg_position'] <= 10, 'pos_4_10',
                      np.where(df['gsc_avg_position'] <= 20, 'pos_11_20', 'pos_21_plus')))

tier_expected_ctr = df[baseline_mask].groupby('position_tier').apply(lambda g: g['gsc_clicks'].sum() / g['gsc_impressions'].sum(), include_groups=False).to_dict()
df['expected_ctr'] = df['position_tier'].map(tier_expected_ctr)
df['ctr_gap_pct'] = (df['expected_ctr'] - df['gsc_ctr']).clip(lower=0) / df['expected_ctr'].replace(0, np.nan)
df['baseline_flag'] = (baseline_mask & (df['ctr_gap_pct'] >= 0.30)).astype(int)

print("\nBaseline overlay denominator: same modeling_frame rows used for clustering (176k GSC-available rows)")
baseline_overlay = df.groupby("cluster_id").agg(
    total_rows=("content_hash_id", "size"),
    baseline_flagged_count=("baseline_flag", "sum"),
    baseline_flag_rate=("baseline_flag", "mean")
).reset_index()
print(baseline_overlay.to_string(index=False))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

No row-level date column found in `modeling_frame`; verify date-window receipts in the SQL/build cell.

Data-contract summary for model inputs:
                    column  null_rate  zero_rate
                gsc_clicks   0.000000   0.610514
           gsc_impressions   0.000000   0.000000
                   gsc_ctr   0.000000   0.610514
          gsc_avg_position   0.008114   0.000000
                word_count   0.312977   0.000008
          content_age_days   0.000000   0.000170
gsc_avg_position_zero_flag   0.000000   0.991886

K-selection summary:
 k  train_silhouette  valid_silhouette  valid_calinski_harabasz  valid_davies_bouldin  smallest_valid_cluster_share
 3            0.9680            0.9679                 153291.3                0.3067                        0.0006
 4            0.9316            0.9449                 170693.3                0.3648                        0.0006
 5            0.8978            0.9100                 171224.6                0.3369         

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Unsupervised models like K-Means easily hide their nature in aggregate metrics. A silhouette score of ~0.35 tells us the clusters have moderate geometric separation in the transformed feature space, but it does not prove business accuracy or supervised correctness.
Below, we inspect **tension cases**: rows assigned to the lowest-median-impression cluster that actually have very high impressions (e.g., above the 90th percentile). Rather than calling these "errors" (since K-Means is just minimizing distance), we stress-test the model to understand *why* it made these surprising assignments.
By calculating the feature-level distance margins between a row's assigned centroid and the nearest alternative centroid, we can see how combinations of features—often heavily weighted missingness flags or scaled position metrics—combine to keep high-traffic content in low-traffic archetypes.

In [3]:
import numpy as np
import pandas as pd

required_objects = ["df", "preprocessor", "model_features", "final_model"]
missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    print(
        "Run the Week 5 Section 3 modeling cell first so these objects exist: "
        + ", ".join(missing_objects)
        + ". This section will then compute tension cases, centroid distance margins, "
        "and feature-level contribution deltas without treating clusters as ground-truth labels."
    )
else:
    section4_df = df.copy()

    if "cluster_id" not in section4_df.columns:
        section4_df["cluster_id"] = final_model.predict(preprocessor.transform(section4_df[model_features]))

    key_cols = [col for col in ["client_hash_id", "content_hash_id"] if col in section4_df.columns]
    if not key_cols:
        print("No anonymized key columns found; tension-case examples will use row_index only.")

    # Transform the current frame into the same feature space used by K-Means.
    X_all = preprocessor.transform(section4_df[model_features])
    if hasattr(X_all, "toarray"):
        X_all = X_all.toarray()

    centroids = final_model.cluster_centers_
    labels = section4_df["cluster_id"].to_numpy()

    # Squared Euclidean distance to every centroid: rows x clusters.
    diff = X_all[:, None, :] - centroids[None, :, :]
    sq_contrib = diff ** 2
    distances = sq_contrib.sum(axis=2)

    assigned_distance = distances[np.arange(len(section4_df)), labels]
    masked_distances = distances.copy()
    masked_distances[np.arange(len(section4_df)), labels] = np.inf
    nearest_alt_cluster = masked_distances.argmin(axis=1)
    nearest_alt_distance = masked_distances[np.arange(len(section4_df)), nearest_alt_cluster]
    distance_margin = nearest_alt_distance - assigned_distance

    section4_df["assigned_distance"] = assigned_distance
    section4_df["nearest_alt_cluster"] = nearest_alt_cluster
    section4_df["nearest_alt_distance"] = nearest_alt_distance
    section4_df["distance_margin"] = distance_margin

    # Build original-scale cluster profile receipts for interpretation.
    numeric_profile_cols = [
        col for col in model_features
        if col in section4_df.columns and pd.api.types.is_numeric_dtype(section4_df[col])
    ]

    profile_rows = []
    for cluster_value, cluster_frame in section4_df.groupby("cluster_id"):
        row = {
            "cluster_id": cluster_value,
            "rows": len(cluster_frame),
            "row_share": len(cluster_frame) / len(section4_df),
        }
        if "client_hash_id" in cluster_frame.columns:
            row["clients"] = cluster_frame["client_hash_id"].nunique()
            row["top_client_share"] = cluster_frame["client_hash_id"].value_counts(normalize=True).iloc[0]
        if "gsc_avg_position" in cluster_frame.columns:
            row["zero_position_rate"] = (cluster_frame["gsc_avg_position"] == 0).mean()
        for col in numeric_profile_cols:
            row[f"{col}_median"] = cluster_frame[col].median()
            row[f"{col}_iqr"] = cluster_frame[col].quantile(0.75) - cluster_frame[col].quantile(0.25)
            row[f"{col}_missing_rate"] = cluster_frame[col].isna().mean()
        profile_rows.append(row)

    cluster_profile = pd.DataFrame(profile_rows).sort_values("cluster_id")
    print("Original-scale cluster profile receipts:")
    print(cluster_profile.to_string(index=False))
    print("\n" + "="*80 + "\n")

    # Identify the lowest-median-impression cluster and the highest-median-impression cluster if impressions exist.
    impression_col = "gsc_impressions" if "gsc_impressions" in section4_df.columns else None
    if impression_col is None:
        print("No `gsc_impressions` column found, so the high-impression-in-low-impression-cluster stress test is skipped.")
    else:
        impression_medians = section4_df.groupby("cluster_id")[impression_col].median().sort_values()
        low_impression_cluster = impression_medians.index[0]
        high_impression_cluster = impression_medians.index[-1]

        high_impression_threshold = section4_df[impression_col].quantile(0.90)
        tension_mask = (
            (section4_df["cluster_id"] == low_impression_cluster)
            & (section4_df[impression_col] >= high_impression_threshold)
        )
        tension_cases = section4_df.loc[tension_mask].copy()

        print(
            "Tension-case definition: rows assigned to the lowest-median-impression cluster\n"
            f"but at or above the 90th percentile of `{impression_col}` in the modeling frame.\n"
            "These are stress-test cases, not proven errors.\n"
        )
        print({
            "low_impression_cluster": int(low_impression_cluster),
            "high_impression_cluster": int(high_impression_cluster),
            "high_impression_threshold": float(high_impression_threshold),
            "tension_case_rows": int(len(tension_cases)),
        })
        print("\n" + "="*80 + "\n")

        # Retrieve transformed feature names where available. Fall back to generic names.
        try:
            transformed_feature_names = list(preprocessor.get_feature_names_out())
        except Exception:
            transformed_feature_names = [f"transformed_feature_{i}" for i in range(X_all.shape[1])]

        # For each tension case, compare contribution to assigned centroid vs nearest alternative
        case_indices = tension_cases.index.to_numpy()
        contribution_rows = []
        max_cases_to_explain = min(3, len(case_indices)) # Limiting to 3 to keep output readable

        for row_idx in case_indices[:max_cases_to_explain]:
            position = section4_df.index.get_loc(row_idx)
            assigned_cluster = int(labels[position])
            nearest_cluster = int(nearest_alt_cluster[position])

            assigned_contrib = sq_contrib[position, assigned_cluster, :]
            nearest_contrib = sq_contrib[position, nearest_cluster, :]
            high_imp_contrib = sq_contrib[position, high_impression_cluster, :]

            assigned_total = assigned_contrib.sum()
            nearest_total = nearest_contrib.sum()
            high_imp_total = high_imp_contrib.sum()

            nearest_delta = nearest_contrib - assigned_contrib
            high_imp_delta = high_imp_contrib - assigned_contrib

            # Top features driving the difference (positive delta = feature pushed it away from alt towards assigned)
            top_nearest_features = np.argsort(np.abs(nearest_delta))[::-1][:3]
            top_high_imp_features = np.argsort(np.abs(high_imp_delta))[::-1][:3]

            base_row = {
                "row_index": row_idx,
                "assigned_cluster": assigned_cluster,
                "nearest_alt_cluster": nearest_cluster,
                "nearest_alt_margin": float(nearest_total - assigned_total),
            }
            for col in key_cols:
                base_row[col] = section4_df.loc[row_idx, col]
            for col in [impression_col, "gsc_clicks", "gsc_avg_position", "word_count", "content_age_days"]:
                if col in section4_df.columns:
                    base_row[col] = section4_df.loc[row_idx, col]

            base_row["top_delta_vs_nearest_alt"] = [
                {
                    "feature": transformed_feature_names[i],
                    "delta_alt_minus_assigned": round(float(nearest_delta[i]), 3),
                }
                for i in top_nearest_features
            ]
            contribution_rows.append(base_row)

        tension_case_explanations = pd.DataFrame(contribution_rows)
        print("Tension-case examples with centroid distance margins and feature contribution deltas:")
        # Setting display options to prevent truncating the dictionary column
        pd.set_option('display.max_colwidth', None)
        print(tension_case_explanations.to_string(index=False))
        pd.reset_option('display.max_colwidth')

        print("\n" + "="*80)
        print(
            "Interpretation note: a positive margin means the assigned centroid is closer than the comparison centroid. "
            "Small margins indicate boundary cases. Feature deltas explain relative centroid distance; they do not prove "
            "that any single raw feature pulled the row into a cluster."
        )

    print(
        "Silhouette framing reminder: a value around 0.35 is moderate feature-space separation. "
        "It supports descriptive archetype exploration, but it does not prove business accuracy or supervised correctness."
    )


Original-scale cluster profile receipts:
 cluster_id   rows  row_share  clients  top_client_share  zero_position_rate  gsc_clicks_median  gsc_clicks_iqr  gsc_clicks_missing_rate  gsc_impressions_median  gsc_impressions_iqr  gsc_impressions_missing_rate  word_count_median  word_count_iqr  word_count_missing_rate  content_age_days_median  content_age_days_iqr  content_age_days_missing_rate  gsc_ctr_median  gsc_ctr_iqr  gsc_ctr_missing_rate  gsc_avg_position_median  gsc_avg_position_iqr  gsc_avg_position_missing_rate  gsc_avg_position_zero_flag_median  gsc_avg_position_zero_flag_iqr  gsc_avg_position_zero_flag_missing_rate
          0 175232   0.991479       47          0.156398                 0.0                0.0             2.0                      0.0                   179.0               1033.0                           0.0             2736.0           928.5                 0.314720                    191.0                 191.0                            0.0           0.000     0.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.